# New Ways Game — Prototype

A figure moves from **bottom-left** to **top-right** on an N×N board.  
Each step is one field: **U / D / L / R**.  
**Rule:** after every move, `find_repeat_seq` applied to the whole move-sequence must return `None`.

## 1 — Core constraint: `find_repeat_seq`

In [ ]:
def find_repeat_seq(s: str) -> str | None:
    """
    Returns the shortest unit U such that s == U * k (k >= 2), or None.
    Examples: 'aa'->'a', 'abab'->'ab', 'abcabc'->'abc', 'abcd'->None
    """
    n = len(s)
    for unit_len in range(1, n // 2 + 1):
        if n % unit_len != 0:
            continue
        unit = s[:unit_len]
        if unit * (n // unit_len) == s:
            return unit
    return None


# Quick sanity check
cases = [('aa','a'), ('abab','ab'), ('abcabc','abc'), ('ababab','ab'),
         ('abcd',None), ('a',None), ('aaaaaa','a'), ('abcabcabc','abc')]
for s, expected in cases:
    got = find_repeat_seq(s)
    mark = '✓' if got == expected else '✗'
    print(f"{mark}  find_repeat_seq({s!r:14}) = {str(got):8}  expected {expected}")

## 2 — Game class

In [ ]:
MOVE_DELTA = {'U': (0, 1), 'D': (0, -1), 'L': (-1, 0), 'R': (1, 0)}

class Game:
    """
    Board is N×N. Origin (0,0) = bottom-left. Target = (N-1, N-1) = top-right.
    Coordinates: (col, row), row increases upward.
    """
    def __init__(self, size: int = 8):
        self.size = size
        self.reset()

    def reset(self):
        self.col, self.row = 0, 0
        self.seq = ''
        self.history = [(0, 0)]
        self.won = False

    @property
    def pos(self):
        return (self.col, self.row)

    @property
    def target(self):
        return (self.size - 1, self.size - 1)

    def valid_moves(self) -> list[str]:
        result = []
        for m, (dc, dr) in MOVE_DELTA.items():
            nc, nr = self.col + dc, self.row + dr
            if 0 <= nc < self.size and 0 <= nr < self.size:
                if find_repeat_seq(self.seq + m) is None:
                    result.append(m)
        return result

    def move(self, m: str) -> tuple[bool, str]:
        dc, dr = MOVE_DELTA[m]
        nc, nr = self.col + dc, self.row + dr
        if not (0 <= nc < self.size and 0 <= nr < self.size):
            return False, 'out of bounds'
        new_seq = self.seq + m
        rep = find_repeat_seq(new_seq)
        if rep is not None:
            return False, f'sequence "{new_seq}" repeats "{rep}"'
        self.col, self.row = nc, nr
        self.seq = new_seq
        self.history.append(self.pos)
        if self.pos == self.target:
            self.won = True
        return True, 'ok'

    def undo(self) -> bool:
        if not self.seq:
            return False
        self.seq = self.seq[:-1]
        self.history.pop()
        self.col, self.row = self.history[-1]
        self.won = False
        return True

    def status(self) -> str:
        if self.won:
            return f'★ WON! ({len(self.seq)} moves)  seq: {self.seq}'
        valid = self.valid_moves()
        if not valid:
            return f'✗ Stuck (no valid moves)  seq: {self.seq or "(none)"}'
        return (f'pos ({self.col},{self.row})  moves: {len(self.seq)}  '
                f'seq: {self.seq or "(none)"}  valid next: {valid}')


# Demo
g = Game(3)
for m in ['R', 'U', 'U', 'R']:
    ok, msg = g.move(m)
    print(f'move {m}: {msg:6}  →  {g.status()}')

## 3 — Board visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def draw_board(game: Game, ax=None, title: str = '') -> plt.Axes:
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    n = game.size
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, n - 0.5)
    ax.set_aspect('equal')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # Checkerboard squares
    for c in range(n):
        for r in range(n):
            color = '#f0d9b5' if (c + r) % 2 == 0 else '#b58863'
            ax.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           color=color, zorder=0))

    # Start (green) and target (gold star)
    ax.add_patch(patches.Circle((0, 0), 0.25, color='#33cc55', alpha=0.8, zorder=1))
    ax.text(n - 1, n - 1, '★', ha='center', va='center',
            fontsize=18, color='gold', zorder=2,
            fontweight='bold', path_effects=[
                __import__('matplotlib.patheffects', fromlist=['withStroke'])
                .withStroke(linewidth=2, foreground='black')])

    # Path trail
    if len(game.history) > 1:
        hist = np.array(game.history, dtype=float)
        ax.plot(hist[:, 0], hist[:, 1], '-',
                color='royalblue', linewidth=2.5, alpha=0.55, zorder=2)

    # Current piece
    ax.add_patch(patches.Circle((game.col, game.row), 0.38,
                                color='royalblue', zorder=3))
    ax.text(game.col, game.row, '♟', ha='center', va='center',
            fontsize=17, color='white', zorder=4)

    if title:
        ax.set_title(title, fontsize=9, pad=6)
    return ax


# Preview
g = Game(8)
for m in ['R','U','R','U','L','U','R','U','R','U']:
    g.move(m)
fig, ax = plt.subplots(figsize=(5, 5))
draw_board(g, ax, title=g.status())
plt.tight_layout()
plt.show()

## 4 — Interactive play (ipywidgets)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def interactive_game(size: int = 8):
    game = Game(size)
    out = widgets.Output()

    def render():
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(5.5, 5.5))
            draw_board(game, ax)
            ax.set_title(game.status(), fontsize=9, wrap=True)
            plt.tight_layout()
            plt.show()
            valid = game.valid_moves()
            print('Valid moves:', valid if valid else 'NONE — stuck!')

    def make_handler(m):
        def handler(b):
            ok, msg = game.move(m)
            if not ok:
                print(f'  ✗ {m} blocked: {msg}')
            render()
        return handler

    W = widgets.Layout(width='64px', height='48px')
    btn_u     = widgets.Button(description='▲ U', layout=W)
    btn_d     = widgets.Button(description='▼ D', layout=W)
    btn_l     = widgets.Button(description='◄ L', layout=W)
    btn_r     = widgets.Button(description='► R', layout=W)
    btn_undo  = widgets.Button(description='Undo',  button_style='warning',
                               layout=widgets.Layout(width='64px'))
    btn_reset = widgets.Button(description='Reset', button_style='danger',
                               layout=widgets.Layout(width='64px'))

    for m, btn in [('U', btn_u), ('D', btn_d), ('L', btn_l), ('R', btn_r)]:
        btn.on_click(make_handler(m))

    def on_undo(b):  game.undo();  render()
    def on_reset(b): game.reset(); render()
    btn_undo.on_click(on_undo)
    btn_reset.on_click(on_reset)

    pad = widgets.Layout(width='64px', height='48px', visibility='hidden')
    controls = widgets.VBox([
        widgets.HBox([widgets.Box(layout=pad), btn_u,    widgets.Box(layout=pad)]),
        widgets.HBox([btn_l,                   btn_d,    btn_r]),
        widgets.HBox([btn_undo,                btn_reset]),
    ])
    display(widgets.HBox([out, controls]))
    render()


interactive_game(size=8)

## 5 — Solver (DFS backtracking with pruning)

In [ ]:
def solve(
    size: int = 8,
    max_depth: int | None = None,
    find_all: bool = False,
    verbose: bool = True,
) -> list[str]:
    """
    DFS backtracking search from (0,0) to (size-1, size-1).

    Pruning:
      - max_depth: abandon paths longer than this
      - Manhattan pruning: remaining steps must cover distance to target

    Returns list of solution sequences found.
    If find_all=False, stops after the first solution.
    """
    if max_depth is None:
        max_depth = 4 * size  # generous heuristic

    game = Game(size)
    solutions: list[str] = []
    nodes = [0]
    tc, tr = game.target

    def dfs():
        nodes[0] += 1

        if game.won:
            solutions.append(game.seq)
            if verbose:
                print(f'  ✓ Solution (len={len(game.seq)}): {game.seq}')
            return not find_all  # True = stop

        remaining = max_depth - len(game.seq)
        manhattan = abs(game.col - tc) + abs(game.row - tr)
        if manhattan > remaining:   # can't reach target in time
            return False

        for m in game.valid_moves():
            game.move(m)
            if dfs():
                return True
            game.undo()
        return False

    if verbose:
        print(f'Searching {size}×{size} board, max_depth={max_depth} ...')
    dfs()
    if verbose:
        print(f'Done — {nodes[0]:,} nodes, {len(solutions)} solution(s) found.')
    return solutions


# --- Run solver ---
sols = solve(size=4, max_depth=20, find_all=False)

In [ ]:
# Visualise a solution
if sols:
    g = Game(4)
    for m in sols[0]:
        g.move(m)
    fig, ax = plt.subplots(figsize=(5, 5))
    draw_board(g, ax, title=f'Solution: {sols[0]}')
    plt.tight_layout()
    plt.show()
else:
    print('No solution found within the depth limit.')

## 6 — Exhaustive search: shortest solution per board size

In [ ]:
def shortest_solution(size: int, hard_limit: int = 200) -> str | None:
    """
    Iterative-deepening DFS: finds the shortest valid solution.
    """
    for depth in range(2 * (size - 1), hard_limit + 1):
        sols = solve(size=size, max_depth=depth, find_all=False, verbose=False)
        if sols:
            print(f'  size={size}: shortest solution len={len(sols[0])}  seq={sols[0]}')
            return sols[0]
    print(f'  size={size}: no solution within depth {hard_limit}')
    return None


print('Shortest solutions by board size:')
for n in range(2, 7):   # adjust upper bound as patience allows
    shortest_solution(n)

## 7 — Random simulation

In [ ]:
import random

def random_game(size: int = 8, seed: int | None = None) -> dict:
    """
    Plays random valid moves until stuck or won.
    Returns a dict with outcome info.
    """
    if seed is not None:
        random.seed(seed)
    g = Game(size)
    while True:
        valid = g.valid_moves()
        if not valid or g.won:
            break
        g.move(random.choice(valid))
    return {'won': g.won, 'steps': len(g.seq), 'seq': g.seq, 'final_pos': g.pos}


def simulate(size: int = 8, n_runs: int = 500, seed: int = 42) -> None:
    random.seed(seed)
    results = [random_game(size) for _ in range(n_runs)]
    wins   = [r for r in results if r['won']]
    losses = [r for r in results if not r['won']]
    print(f'{size}×{size} board — {n_runs} random games:')
    print(f'  Won: {len(wins)} ({100*len(wins)/n_runs:.1f}%)')
    if wins:
        lengths = [r['steps'] for r in wins]
        print(f'  Winning path lengths: min={min(lengths)}, max={max(lengths)}, '
              f'avg={sum(lengths)/len(lengths):.1f}')
        print(f'  Shortest win: {min(wins, key=lambda r: r["steps"])["seq"]}')
    if losses:
        dead_lengths = [r['steps'] for r in losses]
        print(f'  Dead-end path avg length: {sum(dead_lengths)/len(dead_lengths):.1f}')


simulate(size=4, n_runs=2000)
simulate(size=8, n_runs=2000)